In [1]:
import numpy as np
import pandas as pd
from eventflow.data import load_train
from eventflow.validation import assign_folds
from eventflow import gen_metrics as gm

train = load_train()
seqs = train.groupby("sequence_id", sort=False)["event_type"].apply(list)
fold = assign_folds(seqs.index).loc[seqs.index].to_numpy()    

real       = [s[-20:] for s in seqs[fold == 0]]   # last 20 events of 1,300 systems
reference  = [s[-20:] for s in seqs[fold == 1]]   # last 20 events of 1,300 OTHER systems
train_part = list(seqs[fold >= 2])                # 3,900 systems, for the copy checks

rng = np.random.default_rng(2210)
shuffled = [list(rng.permutation(s)) for s in real]   # same events, order destroyed

assert gm.event_js(real, shuffled) < 1e-12                                   # mix: must say "same"
assert gm.ngram_js(real, shuffled, 2) > 5 * gm.ngram_js(real, reference, 2)  # order: must notice
print("Checker works: the mix check ignores order, the bigram check catches it")

pd.DataFrame({
    "real vs real (should be small)": gm.summary(real, reference, train_part),
    "real vs shuffled": gm.summary(real, shuffled, train_part),
}).round(4)

Checker works: the mix check ignores order, the bigram check catches it


,real vs real (should be small),real vs shuffled
event_js,0.0003,0.0000
bigram_js,0.0030,0.0436
trigram_js,0.0402,0.1199
unique_3gram_real,0.0760,0.0760
unique_3gram_fake,0.0754,0.0793
duplicate_rate_real,0.0000,0.0000
duplicate_rate_fake,0.0008,0.0000
copy_rate_real,0.0123,0.0123
copy_rate_fake,0.0131,0.0031
nn_similarity_real,0.7407,0.7407
